In [1]:
# ===================== CONFIG =====================
IMAGE_SIZE = 224
DEVICE_ID = 1
SEED = 42
BATCH_SIZE = 512
EPOCHS = 100
LR = 1e-3
WEIGHT_DECAY = 1e-5
DROPOUT = 0.3

DATA_DIR = r"/home/c/choton/beemachine/datasets/Beemachine_Partwhole_v5/"
LOG_DIR = "./combined_logs/"

CLASSIFIER_NAME = "convnext_nano.in12k"
CLASSIFIER_WEIGHTS = r"/home/c/choton/beemachine/codes/AG_vision_2026/classification/Beemachine/convnext_nano.in12k_timm_new_dataset_logs/checkpoints/best_model.pth"

TRAIN_SHAPE_FEATS = "./predicted_shape_features/beemachine_partwhole_v5_train.csv"
VAL_SHAPE_FEATS   = "./predicted_shape_features/beemachine_partwhole_v5_val.csv"
TEST_SHAPE_FEATS  = "./predicted_shape_features/beemachine_partwhole_v5_test.csv"

In [2]:
# ===================== IMPORTS =====================
import os
import json
import random
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torch.optim.lr_scheduler import ReduceLROnPlateau

import timm
from sklearn.preprocessing import LabelEncoder, StandardScaler

# ===================== SEED =====================
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device(f"cuda:{DEVICE_ID}")

In [3]:
# Load classifier
def load_classifier(name, weights, num_classes, device_id):
    # Define the classification model and load weights
    model = timm.create_model(name, pretrained=False, num_classes=num_classes).to(f"cuda:{device_id}")
    state_dict_c = torch.load(weights, map_location=f"cuda:{device_id}")
    model.load_state_dict(state_dict_c) # Load the classifier state dict
    return model

# --- Replace NaNs within each species group ---
def fill_by_class_mean(df, class_col="species"):
    df = df.replace(0, np.nan)
    df = df.dropna(axis=1, how='all')
    df_numeric = df.select_dtypes(include=[np.number])
    # Fill NaNs in numeric columns using the class-wise mean
    df[df_numeric.columns] = df.groupby(class_col)[df_numeric.columns].transform(
        lambda x: x.fillna(x.mean())
    )
    # Step 2: fill any remaining NaNs globally (for columns that were all NaN in a class)
    df[df_numeric.columns] = df[df_numeric.columns].fillna(df[df_numeric.columns].mean())
    return df

In [4]:
# ===================== LOAD LABELS =====================
train_df = pd.read_csv(os.path.join(DATA_DIR, "train_aug_labels.csv"))
val_df   = pd.read_csv(os.path.join(DATA_DIR, "val_labels.csv"))
test_df  = pd.read_csv(os.path.join(DATA_DIR, "test_labels.csv"))

le = LabelEncoder()
train_df["label"] = le.fit_transform(train_df["species"])
val_df["label"]   = le.transform(val_df["species"])
test_df["label"]  = le.transform(test_df["species"])
num_classes = len(le.classes_)
print(f"Shape of label dataframes: train={train_df.shape}, val={val_df.shape}, test={test_df.shape}")
print("Head of train label dataframe:")
train_df.head(2)

Shape of label dataframes: train=(34722, 3), val=(1158, 3), test=(771, 3)
Head of train label dataframe:


,image,species,label
0,Bombus_distinguendus_GBIF_iNat_2573822774_2-jp...,Bombus_distinguendus,36
1,Bombus_bifarius_47362646_1_1_jpg.rf.bd7d10af8d...,Bombus_bifarius,14


In [5]:
# ===================== LOAD SHAPE FEATURES =====================
train_shape_df = pd.read_csv(TRAIN_SHAPE_FEATS)
val_shape_df   = pd.read_csv(VAL_SHAPE_FEATS)
test_shape_df  = pd.read_csv(TEST_SHAPE_FEATS)

train_df = fill_by_class_mean(pd.merge(train_shape_df, train_df, on="image"))
val_df   = fill_by_class_mean(pd.merge(val_shape_df,   val_df,   on="image"))
test_df  = fill_by_class_mean(pd.merge(test_shape_df,  test_df,  on="image"))
print(f"Shape of shape_feats dataframes: train={train_df.shape}, val={val_df.shape}, test={test_df.shape}")
print("Head of train shape_feats dataframe:")
train_df.head(2)

Shape of shape_feats dataframes: train=(34722, 940), val=(1158, 940), test=(771, 940)
Head of train shape_feats dataframe:


,image,head_area,head_perimeter,head_aspect_ratio,head_extent,head_solidity,head_eccentricity,head_orientation,head_circularity,head_elongation,...,full_brisque,full_niqe,full_piqe,head_to_thorax_area,thorax_to_abdomen_area,head_to_total_area,thorax_to_total_area,abdomen_to_total_area,species,label
0,Bombus_distinguendus_GBIF_iNat_2573822774_2-jp...,3925.0,368.46910,1.107346,0.588015,0.772942,0.429514,-1.047061,0.363285,0.096940,...,58.42423,30.060743,43.24497,0.175741,2.572449,0.112332,0.639192,0.248476,Bombus_distinguendus,36.0
1,Bombus_bifarius_47362646_1_1_jpg.rf.bd7d10af8d...,865.0,129.49748,2.422290,0.706699,0.929109,0.910807,1.360611,0.648192,0.587168,...,94.43765,26.321207,87.51460,3.728448,0.028823,0.094577,0.025366,0.880057,Bombus_bifarius,14.0


In [6]:
X_train = train_df.drop(columns=["image", "species", "label"]).values
X_val   = val_df.drop(columns=["image", "species", "label"]).values
X_test  = test_df.drop(columns=["image", "species", "label"]).values

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

y_train = train_df["label"].values
y_val   = val_df["label"].values
y_test  = test_df["label"].values

In [7]:
# ===================== DATASET =====================
class CombinedDataset(Dataset):
    def __init__(self, df, img_dir, X, y):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.transform = transforms.Compose([
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=(0.4925, 0.4475, 0.3490),
                std=(0.2392, 0.2265, 0.2213)
            )
        ])
        self.num_classes = self.df["label"].nunique()

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img = Image.open(
            os.path.join(self.img_dir, self.df.loc[idx, "image"])
        ).convert("RGB")
        img = self.transform(img)
        return img, self.X[idx], self.y[idx]

In [22]:
# ===================== LOADERS FOR FEATURE EXTRACTION =====================
train_img_dir = os.path.join(DATA_DIR, "train", "aug_images")
val_img_dir   = os.path.join(DATA_DIR, "valid", "images")
test_img_dir  = os.path.join(DATA_DIR, "test",  "images")

train_ds = CombinedDataset(train_df, train_img_dir, X_train, y_train)
val_ds   = CombinedDataset(val_df,   val_img_dir,   X_val,   y_val)
test_ds  = CombinedDataset(test_df,  test_img_dir,  X_test,  y_test)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=False)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

num_classes = train_ds.num_classes
print(f"Total classes = Train: {train_ds.num_classes} | Val: {val_ds.num_classes} | Test: {test_ds.num_classes}")
print(f"Total images = Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

Total classes = Train: 160 | Val: 114 | Test: 113
Total images = Train: 34722 | Val: 1158 | Test: 771


In [23]:
# Loading timm model
device = torch.device(f"cuda:{DEVICE_ID}")
model = load_classifier(
            name=CLASSIFIER_NAME, 
            weights=CLASSIFIER_WEIGHTS, 
            num_classes=num_classes, 
            device_id=DEVICE_ID
        )
model.to(device)
model.eval()
for p in model.parameters():
    p.requires_grad = False

vis_dim = model.num_features
print("Number of visual features from classification model:", vis_dim)
model

Number of visual features from classification model: 640


ConvNeXt(
  (stem): Sequential(
    (0): Conv2d(3, 80, kernel_size=(4, 4), stride=(4, 4))
    (1): LayerNorm2d((80,), eps=1e-06, elementwise_affine=True)
  )
  (stages): Sequential(
    (0): ConvNeXtStage(
      (downsample): Identity()
      (blocks): Sequential(
        (0): ConvNeXtBlock(
          (conv_dw): Conv2d(80, 80, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=80)
          (norm): LayerNorm2d((80,), eps=1e-06, elementwise_affine=True)
          (mlp): Mlp(
            (fc1): Conv2d(80, 320, kernel_size=(1, 1), stride=(1, 1))
            (act): GELU()
            (drop1): Dropout(p=0.0, inplace=False)
            (norm): Identity()
            (fc2): Conv2d(320, 80, kernel_size=(1, 1), stride=(1, 1))
            (drop2): Dropout(p=0.0, inplace=False)
          )
          (shortcut): Identity()
          (drop_path): Identity()
        )
        (1): ConvNeXtBlock(
          (conv_dw): Conv2d(80, 80, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=80)


In [24]:
@torch.no_grad()
def extract_visual_features(model, loader):
    feats, labels = [], []
    for imgs, _, y in tqdm(loader, desc="Extracting visual features"):
        imgs = imgs.to(device)
        f = model.forward_features(imgs)
        f = model.head.global_pool(f)
        f = model.head.norm(f)
        f = model.head.flatten(f)
        feats.append(f.cpu())
        labels.append(y)
    return torch.cat(feats), torch.cat(labels)

In [28]:
train_vis, y_train_t = extract_visual_features(model, train_loader)
val_vis,   y_val_t   = extract_visual_features(model, val_loader)
test_vis,  y_test_t  = extract_visual_features(model, test_loader)

Extracting visual features: 100%|██████████| 2/2 [01:08<00:00, 34.29s/it]


In [29]:
# ===================== CONCAT FEATURES =====================
X_train_combined = torch.cat([torch.tensor(X_train), train_vis], dim=1)
X_val_combined   = torch.cat([torch.tensor(X_val),   val_vis],   dim=1)
X_test_combined  = torch.cat([torch.tensor(X_test),  test_vis],  dim=1)

In [37]:
X_val_combined.shape

torch.Size([1158, 1577])

In [38]:
X_val_combined.shape

torch.Size([1158, 1577])

In [39]:
# ===================== FINAL DATASET =====================
class FeatureDataset(Dataset):
    def __init__(self, X, y):
        self.X = X.float()
        self.y = y.long()
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]
    
train_dataset = FeatureDataset(X_train_combined, y_train_t)
val_dataset = FeatureDataset(X_val_combined, y_val_t)
test_dataset = FeatureDataset(X_test_combined, y_test_t)

train_loader = DataLoader(train_dataset,
                          batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,
                          batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset,
                          batch_size=BATCH_SIZE)

In [40]:
class MLPClassifier(nn.Module):
    def __init__(self, input_dim, num_classes, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.LayerNorm(512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes)
        )
        # simple init
        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.net(x)

In [45]:
X_train_combined.shape[1]

1577

In [46]:
device = torch.device(f"cuda:{DEVICE_ID}")

input_dim = X_train_combined.shape[1]
model = MLPClassifier(input_dim, num_classes).to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

In [47]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for X, y in loader:
        X, y = X.to(device), y.to(device)

        optimizer.zero_grad()
        outputs = model(X)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * X.size(0)

        preds = outputs.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)

    return total_loss / total, correct / total

def validate_one_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            outputs = model(X)
            loss = criterion(outputs, y)

            total_loss += loss.item() * X.size(0)

            preds = outputs.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)

    return total_loss / total, correct / total

def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler,
                device, epochs, log_dir):
    # Full training loop with logging and checkpointing 
    history = {"epoch": [], "train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    checkpoints_dir = os.path.join(log_dir, "checkpoints")
    os.makedirs(checkpoints_dir, exist_ok=True)

    # Initialize validation loss and start training
    best_val_loss = float("inf")
    print("Starting the first epoch...")

    for epoch in range(epochs):
        # ---- Training ----
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)

        # ---- Validation ----
        val_loss, val_acc = validate_one_epoch(model, val_loader, criterion, device)

        # ---- Scheduler ----
        scheduler.step(val_loss)

        # ---- Logging ----
        history["epoch"].append(epoch + 1)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        # ---- Checkpoint ----
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_path = os.path.join(checkpoints_dir, "best_model.pth")
            torch.save(model.state_dict(), best_path)

            checkpoint_path = os.path.join(checkpoints_dir, f"checkpoint_epoch{epoch+1}.pth")
            torch.save({
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_loss": val_loss
            }, checkpoint_path)
        
        print(f"\r Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")    
    print("Training complete!")

    # ---- Save logs ----
    df = pd.DataFrame(history)
    csv_logfile = os.path.join(log_dir, "training_log.csv")
    json_logfile = os.path.join(log_dir, "training_log.json")

    df.to_csv(csv_logfile, index=False)
    with open(json_logfile, "w") as f:
        json.dump(history, f, indent=4)

    return history

In [48]:
%%time

history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    epochs=EPOCHS,
    log_dir=LOG_DIR
)

Starting the first epoch...
 Epoch 1/100 | Train Loss: 1.7258 | Train Acc: 0.8191 | Val Loss: 2.5767 | Val Acc: 0.6088
 Epoch 2/100 | Train Loss: 1.0117 | Train Acc: 0.9968 | Val Loss: 2.4664 | Val Acc: 0.6183
 Epoch 3/100 | Train Loss: 0.9644 | Train Acc: 0.9998 | Val Loss: 2.4795 | Val Acc: 0.6192
 Epoch 4/100 | Train Loss: 0.9436 | Train Acc: 0.9999 | Val Loss: 2.4961 | Val Acc: 0.6123
 Epoch 5/100 | Train Loss: 0.9309 | Train Acc: 1.0000 | Val Loss: 2.4667 | Val Acc: 0.6149
 Epoch 6/100 | Train Loss: 0.9214 | Train Acc: 1.0000 | Val Loss: 2.4697 | Val Acc: 0.6200
 Epoch 7/100 | Train Loss: 0.9161 | Train Acc: 1.0000 | Val Loss: 2.4775 | Val Acc: 0.6174
 Epoch 8/100 | Train Loss: 0.9117 | Train Acc: 1.0000 | Val Loss: 2.4864 | Val Acc: 0.6166
 Epoch 9/100 | Train Loss: 0.9078 | Train Acc: 1.0000 | Val Loss: 2.4797 | Val Acc: 0.6183
 Epoch 10/100 | Train Loss: 0.9062 | Train Acc: 1.0000 | Val Loss: 2.4845 | Val Acc: 0.6192
 Epoch 11/100 | Train Loss: 0.9050 | Train Acc: 1.0000 | Val 

-----------------------------------------